# Check-in 3: the core watermark benchmark
CMPE 261, Group 6. TR, RI, SFW, GS, WIND, PRC on pinned SD 2.1-base.

This notebook prepares a fresh dataset and retains historical runs unchanged. Full runs require a passing six-scheme GPU preflight and immutable inputs, environment, and key state. No sample counts, attack strengths, or budgets are reduced automatically.

Run each cell in order. The full queue is serial. Do not run a second worker against the same output directory. Interrupting an experiment is safe; reopen this notebook and rerun the same command with the same source and environment to resume.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CMPE261 Group 6 - Forging the Watermark')
assert DRIVE_ROOT.is_dir(), 'Use the existing project Drive folder; do not create a replacement.'
OUT = DRIVE_ROOT / 'checkin3'
PILOT = DRIVE_ROOT / 'checkin3-preflight-v1'
SOURCE_ZIP = DRIVE_ROOT / 'checkin3-source.zip'
REPO = Path('/content/cmpe261-watermark-forgery-checkin3')
BRANCH = 'checkin3-core-benchmark'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
print('Drive:', DRIVE_ROOT, '| full results:', OUT, '| independent pilot:', PILOT)


## Source
Keep `checkin3-source.zip` in the existing project Drive folder. After publication, the clone path below can be used instead. Preserve the exact source archive and saved environment when resuming.


In [ ]:
import zipfile
if SOURCE_ZIP.exists():
    if REPO.exists() and any(REPO.iterdir()):
        print('Preserving existing source; the environment and pilot gates will verify it.')
    else:
        REPO.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            archive.extractall(REPO)
else:
    if not REPO.exists():
        subprocess.run(['git','clone','--branch',BRANCH,
                        'https://github.com/yashashav-dk/cmpe261-watermark-forgery.git',str(REPO)],check=True)
assert (REPO/'src/wmforge/checkin3.py').exists(), 'Upload the Check-in 3 source archive.'
os.chdir(REPO)
print('Source:', REPO)


## Isolated Python 3.12 and recorded GPU stack
The notebook kernel may use another Python version. All experiments below run in a separate Python 3.12 environment. Setup uses checked subprocess calls and stops on errors. The complete compatible installed environment is captured after `pip check`; transitive versions are then bound into the run manifest. Reuse the captured environment for a disconnected worker.


In [ ]:
import json, re
setup = DRIVE_ROOT/'checkin3-setup'
setup.mkdir(exist_ok=True)
os.environ.update(MPLBACKEND='Agg', CUBLAS_WORKSPACE_CONFIG=':4096:8', PYTHONPATH=str(REPO/'src'))
PY = Path('/content/checkin3-py312/bin/python')
saved_freeze = setup/'environment.txt'
saved_pilot = PILOT/'preflight.json'
pilot = json.loads(saved_pilot.read_text()) if saved_pilot.exists() else None
python_version = '3.12'
if (setup/'python.txt').exists():
    python_version = re.search(r'3\.12\.\d+', (setup/'python.txt').read_text()).group()
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','python','install',python_version],check=True)
if not PY.exists():
    subprocess.run([sys.executable,'-m','uv','venv','--seed','--python',python_version,str(PY.parent.parent)],check=True)
if saved_freeze.exists():
    restore_pins = setup/'requirements-restore.txt'
    restore_pins.write_text('\n'.join(line for line in saved_freeze.read_text().splitlines() if not line.lower().startswith('wmforge') and not line.startswith('-e '))+'\n')
    command = [str(PY),'-m','pip','install','--extra-index-url','https://download.pytorch.org/whl/cu128','-r',str(restore_pins)]
    if pilot:
        command.append('pip=='+pilot['environment']['pip'])
    subprocess.run(command,cwd=REPO,check=True)
    subprocess.run([str(PY),'-m','pip','install','--no-deps','--force-reinstall',str(REPO)],cwd=REPO,check=True)
else:
    subprocess.run([str(PY),'-m','pip','install','torch==2.10.0','torchvision==0.25.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True)
    subprocess.run([str(PY),'-m','pip','install','-r','requirements-checkin3.txt'],cwd=REPO,check=True)
    subprocess.run([str(PY),'-m','pip','install','-e','.[dev]'],cwd=REPO,check=True)
subprocess.run([str(PY),'-m','pip','check'],check=True)
subprocess.run([str(PY),'-c','from wmforge.checkin3 import check_environment; from ldpc import bp_decoder; print(check_environment())'],cwd=REPO,check=True)
freeze = subprocess.check_output([str(PY),'-m','pip','freeze'],text=True)
if saved_freeze.exists():
    def third_party_pins(text):
        return sorted(line for line in text.splitlines() if line and not line.startswith(('#','-e ')) and not line.lower().startswith('wmforge'))
    assert third_party_pins(saved_freeze.read_text())==third_party_pins(freeze), 'Environment differs from saved setup.'
    restored_freeze = setup/'environment-restored.txt'
    if restored_freeze.exists():
        assert restored_freeze.read_text()==freeze, 'Restored environment differs.'
    else:
        restored_freeze.write_text(freeze)
else:
    saved_freeze.write_text(freeze)
if not (setup/'python.txt').exists():
    (setup/'python.txt').write_text(subprocess.check_output([str(PY),'--version'],text=True))
(setup/'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'],text=True))
if pilot:
    command = "from wmforge.checkin3 import check_environment,source_identity; import json; from pathlib import Path; p=json.loads(Path("+repr(str(saved_pilot))+").read_text()); assert check_environment()==p['environment']; assert source_identity()==p['source']; print('Exact saved pilot environment and source verified')"
    subprocess.run([str(PY),'-c',command],cwd=REPO,check=True)
print('Environment ready:',PY)


## Tests and deterministic input extension
Select 200 unique prompts and 100 unique COCO photographs with seed 261. Preserve the committed 50-item selections as exact prefixes, including the first five forgery covers. Selection refuses to overwrite different existing files. Save the selections on Drive and commit these exact files with the measurements after the run.


In [ ]:
subprocess.run([str(PY),'-m','pytest','-q','-m','not slow'],check=True)
subprocess.run([str(PY),'scripts/select_checkin3_data.py'],check=True)
import shutil
for name in ['prompts_checkin3.txt','coco_val2017_checkin3.txt','checkin3-selection.json']:
    source=REPO/'data'/name
    destination=setup/name
    if destination.exists():
        assert destination.read_bytes()==source.read_bytes(), 'Committed selections changed.'
    else:
        shutil.copy2(source,destination)
print('Selections saved; preserve and version these files.')

(setup/'verification-status.json').write_text(json.dumps(dict(status='ready',tests='pytest -q -m not slow',selections='committed prefixes and deterministic extension verified')))


## GPU preflight and measured cost
This pilot is separate from the benchmark. It exercises all six schemes, four removals, score finiteness, key restoration, image dimensions, LPIPS, 10 imprint steps, checkpoint replay, GPU memory, and Drive save/reopen. A pilot directory containing evidence is immutable: inspect an existing successful preflight rather than rerunning it. Choose a new numbered pilot directory only for a deliberately new pilot.

Inspect `preflight.json` and actual GPU cost before the full queue. The estimated imprint hours exclude baseline and removal cost; the historical 16–20-hour total on an RTX PRO 6000 is a planning estimate only. A T4 may have insufficient memory for the full-precision workload. Failed preflight blocks full execution.


In [ ]:
if (PILOT/'preflight.json').exists():
    import json
    pilot=json.loads((PILOT/'preflight.json').read_text())
    print(json.dumps(pilot,indent=2))
    assert pilot['passed'], 'Inspect failure; fix the environment, then use a new pilot directory.'
else:
    subprocess.run([str(PY),'-m','wmforge.preflight','--out',str(PILOT)],check=True)


## Full F1, F2, F3
F1: 100 watermarked + 200 unwatermarked generations + 100 real photos per scheme. Inclusion uses the native detector only: clean TPR ≥90%, generated FPR ≤5%.

Calibration uses only the 200 unwatermarked scores, oriented so larger values are stronger evidence, with a strict threshold allowing at most two positives. This is an empirical operating point, with 0.5-percentage-point resolution. PRC's score-only calibrated test is separate from its native decoder verdict.

F2: same 10 covers per included scheme, 150 steps per cover, checks every 10, no stopping at detection. Primary rates and PSNR/SSIM/LPIPS use the saved images at steps 50/100/150.

F3: JPEG quality 95, Gaussian blur radius 2, centered crop ratio 0.8 and resize, and SD 2.1-base purification strength 0.3 with empty prompt. Apply to every one of the 100 saved watermarked images, including initial detector misses. All decisions use saved artifacts. Never run these cells concurrently.

The next cell supervises the full serial F1 → F2 → F3 queue, saves dated stage logs and VM boot identity, and reports durable progress every 30 seconds. Leave it running. If Colab replaces the VM, restore the captured environment and rerun the same cell to resume. Do not start a second worker.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import os, json, subprocess, time, fcntl
DRIVE_ROOT = Path('/content/drive/MyDrive/CMPE261 Group 6 - Forging the Watermark')
REPO = Path('/content/cmpe261-watermark-forgery-checkin3')
PY = Path('/content/checkin3-py312/bin/python')
PILOT = DRIVE_ROOT/'checkin3-preflight-v1'
OUT = DRIVE_ROOT/'checkin3'
setup = DRIVE_ROOT/'checkin3-setup'
os.environ.update(MPLBACKEND='Agg', CUBLAS_WORKSPACE_CONFIG=':4096:8', PYTHONPATH=str(REPO/'src'))
boot_id = Path('/proc/sys/kernel/random/boot_id').read_text().strip()
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
queue_status = setup/'queue-status.json'
queue_lock = (setup/'queue.lock').open('a')
fcntl.flock(queue_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
def save_status(value):
    partial = queue_status.with_suffix('.json.part')
    partial.write_text(json.dumps(value, indent=2))
    os.replace(partial, queue_status)
def progress():
    path = OUT/'measurements-v2.json'
    if not path.exists():
        return {'saved_measurements':0,'saved_inputs':len(list((OUT/'inputs').glob('*.png')))}
    rows = json.loads(path.read_text())['rows']
    counts = Counter((r['stage'],r['scheme'],r['kind'],r['attack']) for r in rows)
    return {'saved_measurements':len(rows),'groups':{'/'.join(k):v for k,v in sorted(counts.items())}}
BASE = [str(PY),'-u','-m','wmforge.checkin3','--out',str(OUT),'--preflight',str(PILOT/'preflight.json')]
queue_ok = True
child = None
try:
    assert json.loads((PILOT/'preflight.json').read_text())['passed']
    assert json.loads((setup/'verification-status.json').read_text())['status']=='ready'
    for stage in ['baseline','imprint','removal']:
        path = setup/(stamp+'-'+stage+'.log')
        with path.open('w') as handle:
            child = subprocess.Popen(BASE+['--stages',stage], cwd=REPO, stdout=handle, stderr=subprocess.STDOUT, env=os.environ.copy())
            status = dict(status='running',stage=stage,pid=child.pid,boot_id=boot_id,started_utc=datetime.now(timezone.utc).isoformat(),log=str(path),out=str(OUT))
            save_status(status)
            print('START',json.dumps(status),flush=True)
            while child.poll() is None:
                print(datetime.now(timezone.utc).isoformat(),json.dumps(progress()),flush=True)
                time.sleep(30)
            status.update(status='complete' if child.returncode==0 else 'failed',returncode=child.returncode,finished_utc=datetime.now(timezone.utc).isoformat(),progress=progress())
            save_status(status)
        print('END',stage,'EXIT',child.returncode,json.dumps(progress()),flush=True)
        print(path.read_text()[-2500:],flush=True)
        if child.returncode:
            queue_ok = False
            break
    if queue_ok:
        for command in [
            [str(PY),'-m','wmforge.checkin3_report','--run',str(OUT)],
            [str(PY),'scripts/checkin3_draft.py','--run',str(OUT),'--out',str(OUT/'Group6_CMPE261_CheckIn_3_draft.pdf')],
        ]:
            result = subprocess.run(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            print(result.stdout[-5000:],flush=True)
            if result.returncode:
                queue_ok=False
                status.update(status='report-failed',returncode=result.returncode)
                break
        if queue_ok:
            status.update(status='complete',stage='all',finished_utc=datetime.now(timezone.utc).isoformat(),progress=progress())
        save_status(status)
except KeyboardInterrupt:
    if child is not None and child.poll() is None:
        child.send_signal(2)
        child.wait()
    save_status(dict(status='interrupted',boot_id=boot_id,progress=progress(),finished_utc=datetime.now(timezone.utc).isoformat()))
    print('Interrupted; saved artifacts and checkpoints remain available for resume.',flush=True)
except Exception as error:
    save_status(dict(status='failed',error=str(error),boot_id=boot_id,progress=progress(),finished_utc=datetime.now(timezone.utc).isoformat()))
    print('QUEUE FAILED:',str(error),flush=True)
    raise
finally:
    fcntl.flock(queue_lock, fcntl.LOCK_UN)
    queue_lock.close()
print('Full queue status:',queue_status.read_text(),flush=True)


In [ ]:
assert json.loads((setup/'queue-status.json').read_text())['status']=='complete', 'Inspect the serial queue status before continuing.'


In [ ]:
print('Saved artifacts:', OUT)
print('Queue evidence:', setup/'queue-status.json')


## Progress, reporting, backups
The running queue prints counts from the saved measurement log. After it finishes, the following cell checks durable status and renders the report. A VM replacement can leave the last printed output stale: compare the saved boot identity and current runtime before resuming.


In [ ]:
import json
from collections import Counter
status_path = setup/'queue-status.json'
print(status_path.read_text() if status_path.exists() else 'Queue has not started.')
log_path = OUT/'measurements-v2.json'
if log_path.exists():
    rows = json.loads(log_path.read_text())['rows']
    print('Saved measurements:',len(rows))
    print(dict(Counter((r['stage'],r['scheme'],r['attack'],r['kind']) for r in rows)))
if status_path.exists() and json.loads(status_path.read_text())['status']=='complete':
    subprocess.run([str(PY),'-m','wmforge.checkin3_report','--run',str(OUT)],cwd=REPO,check=True)
    from IPython.display import Markdown,display
    display(Markdown((OUT/'summary-v2.md').read_text()))


Run backups only after the worker has stopped, so the archive contains a consistent set of log, checkpoints, keys, and measured images. Keep key material in the team Drive; publish only non-secret manifests, selections, results, and report artifacts.


In [ ]:
from datetime import datetime,timezone
# A nonblocking lock refuses to archive during an active write.
from wmforge.checkin3 import single_writer
with single_writer(OUT):
    backup=DRIVE_ROOT/('checkin3-backup-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    shutil.make_archive(str(backup),'zip',root_dir=OUT)
print('Backup:',str(backup)+'.zip')


## Draft report and final checks
Generate the draft with the verified summary, inspect every PDF page, confirm member contributions and instructor/TA access, and review partial work explicitly. The report generator labels absent results as pending. Repository publication and Canvas submission are separate actions.


In [ ]:
subprocess.run([str(PY),'scripts/checkin3_draft.py','--run',str(OUT),
                '--out',str(OUT/'Group6_CMPE261_CheckIn_3_draft.pdf')],check=True)
print('Draft:',OUT/'Group6_CMPE261_CheckIn_3_draft.pdf')
